# 🌿 D2C Skincare E-Commerce: Аналитический проект

**Автор:** Начинающий аналитик данных  
**Дата:** 2025  
**Инструменты:** Python · Pandas · Matplotlib · Seaborn  
**Данные:** Синтетический датасет D2C-бренда косметики (500 клиентов, 1250 заказов, 28 товаров)

---

> *Этот проект — моя первая полноценная аналитическая работа. Я постаралась не просто показать умение работать с кодом, но и сделать выводы, которые могли бы быть полезны реальному бизнесу.*

## 📋 Содержание

1. [Постановка бизнес-задачи](#task)
2. [Импорт библиотек](#imports)
3. [Загрузка данных](#load)
4. [Первичный осмотр данных (EDA — часть 1)](#eda1)
5. [Очистка и предобработка](#cleaning)
6. [Исследовательский анализ (EDA — часть 2)](#eda2)
7. [Углублённая аналитика](#advanced)
8. [Ключевые инсайты](#insights)
9. [Бизнес-рекомендации](#recommendations)
10. [Заключение и следующие шаги](#conclusion)


---
<a id='task'></a>
## 1. 🎯 Постановка бизнес-задачи и цели анализа

Представьте небольшой D2C-бренд косметики — он продаёт уходовую косметику напрямую покупателям через сайт, мобильное приложение и другие каналы. У бренда уже 500 клиентов, есть история заказов, возвратов и отзывов.

Перед нами стоят классические вопросы, которые задаёт себе любой продуктовый аналитик:

**Главные вопросы:**
- 💰 Как меняется выручка со временем, и есть ли сезонность?
- 🛍️ Какие товары и категории приносят больше всего денег?
- 👩 Кто наши самые ценные клиентки — и как их удержать?
- 📦 Почему клиенты возвращают товары — и с какими продуктами это случается чаще?
- 📣 Какой канал привлечения приводит лучших клиентов?

**Что я буду делать:**
1. Очищу и объединю 7 таблиц данных в единую аналитическую базу
2. Проведу визуальный анализ ключевых метрик
3. Сегментирую клиентов с помощью RFM-анализа
4. Проанализирую возвраты и отзывы
5. Сформулирую конкретные рекомендации для бизнеса

> ⚠️ **Важное замечание:** Датасет синтетический и создан для учебных целей. Все выводы носят демонстрационный характер и требуют проверки на реальных данных.


---
<a id='imports'></a>
## 2. 📦 Импорт библиотек и настройка окружения


In [ ]:
# Стандартные библиотеки для работы с данными
import pandas as pd
import numpy as np

# Библиотеки для визуализации
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Для предупреждений
import warnings
warnings.filterwarnings('ignore')

# --- Настройка стиля визуализаций ---
# Я выбрала seaborn-v0_8 как базовый стиль — он выглядит чище, чем дефолтный matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')

# Фиксируем размеры фигур по умолчанию
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# Чтобы русский текст отображался корректно на графиках (если система поддерживает)
plt.rcParams['axes.unicode_minus'] = False

print("✅ Все библиотеки успешно загружены!")
print(f"   Pandas версия: {pd.__version__}")
print(f"   NumPy версия: {np.__version__}")


---
<a id='load'></a>
## 3. 📂 Загрузка данных

У нас 6 CSV-файлов. Каждый — это отдельная таблица реляционной базы данных.  
Важно понять, как они связаны между собой — это ключ к правильному анализу.

**Схема связей:**
```
customers ──→ orders ──→ order_items ──→ products
                │
                ├──→ returns ──→ products
                └──→ reviews ──→ products
```


In [ ]:
# =====================================================================
# Загружаем все таблицы из CSV-файлов
# Указываем пути — у меня данные лежат рядом с ноутбуком в папке data/
# =====================================================================

DATA_PATH = 'data/'  # <-- измените путь, если у вас другая структура

customers  = pd.read_csv(DATA_PATH + 'Customers.csv')
orders     = pd.read_csv(DATA_PATH + 'Orders.csv')
order_items= pd.read_csv(DATA_PATH + 'Order_Items.csv')
products   = pd.read_csv(DATA_PATH + 'Products.csv')
returns    = pd.read_csv(DATA_PATH + 'Returns.csv')
reviews    = pd.read_csv(DATA_PATH + 'Reviews.csv')

print("✅ Данные загружены! Вот размеры каждой таблицы:\n")
tables = {
    'customers':   customers,
    'orders':      orders,
    'order_items': order_items,
    'products':    products,
    'returns':     returns,
    'reviews':     reviews,
}
for name, df in tables.items():
    print(f"  📄 {name:<15} → {df.shape[0]:>5} строк, {df.shape[1]:>2} столбцов")


---
<a id='eda1'></a>
## 4. 🔍 Первичный осмотр данных

Прежде чем что-то считать, я хочу «познакомиться» с каждой таблицей:  
посмотреть на типы данных, пропуски, примеры значений.  
Это как открыть новую книгу — сначала листаешь, потом читаешь 📖


In [ ]:
# ================================================================
# Смотрим на каждую таблицу по очереди
# Функция-помощник для быстрого обзора
# ================================================================

def quick_overview(df, name):
    print(f"{'='*60}")
    print(f"📋 Таблица: {name.upper()}")
    print(f"{'='*60}")
    print(f"Размер: {df.shape[0]} строк × {df.shape[1]} столбцов\n")
    print("Первые 3 строки:")
    display(df.head(3))
    print("\nТипы данных и пропуски:")
    info = pd.DataFrame({
        'dtype': df.dtypes,
        'non_null': df.notna().sum(),
        'null_count': df.isna().sum(),
        'null_%': (df.isna().mean() * 100).round(1)
    })
    display(info)
    print()

# Смотрим все таблицы
for name, df in tables.items():
    quick_overview(df, name)


In [ ]:
# ================================================================
# Проверка на дубликаты в каждой таблице
# ================================================================

print("🔍 Проверка дубликатов:\n")
for name, df in tables.items():
    dups = df.duplicated().sum()
    status = "✅ нет" if dups == 0 else f"⚠️ найдено {dups}"
    print(f"  {name:<15} → дубликаты: {status}")


In [ ]:
# ================================================================
# Базовая статистика по числовым столбцам (только orders и products)
# ================================================================

print("📊 Статистика по заказам:")
display(orders[['gross_amount', 'discount_amount', 'shipping_fee', 'final_amount']].describe().round(2))

print("\n📊 Статистика по товарам:")
display(products[['mrp', 'cost_price', 'stock_qty']].describe().round(2))


---
<a id='cleaning'></a>
## 5. 🧹 Очистка и предобработка данных

После первичного осмотра я вижу, что основные проблемы:
1. Даты хранятся как строки — нужно перевести в `datetime`
2. Некоторые категориальные столбцы лучше сделать типом `category`
3. Нужно создать новые полезные признаки (месяц, год, сезон и т.д.)

> *Примечание: я пробовала разные форматы дат — в файлах формат DD-MM-YYYY, это важно указать явно при парсинге, иначе pandas может перепутать день и месяц.*


In [ ]:
# ================================================================
# Шаг 1: Преобразование дат во всех таблицах
# Формат в файлах: DD-MM-YYYY
# ================================================================

date_cols = {
    'customers':   ['signup_date'],
    'orders':      ['order_date', 'delivered_date'],
    'products':    ['launch_date'],
    'returns':     ['return_date'],
    'reviews':     ['review_date'],
}

for table_name, cols in date_cols.items():
    df = tables[table_name]
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], dayfirst=True, errors='coerce')
    tables[table_name] = df

# Обновляем переменные
customers   = tables['customers']
orders      = tables['orders']
order_items = tables['order_items']
products    = tables['products']
returns     = tables['returns']
reviews     = tables['reviews']

print("✅ Даты успешно преобразованы!")
print(f"   Пример: orders.order_date → {orders['order_date'].dtype}")


In [ ]:
# ================================================================
# Шаг 2: Категориальные столбцы → тип 'category'
# Это экономит память и ускоряет group by
# ================================================================

cat_cols_customers = ['gender', 'age_group', 'acquisition_channel', 'city', 'state']
cat_cols_orders    = ['order_status', 'payment_method', 'sales_channel']
cat_cols_products  = ['category', 'skin_type']
cat_cols_returns   = ['return_reason', 'refund_status']

for col in cat_cols_customers:
    customers[col] = customers[col].astype('category')

for col in cat_cols_orders:
    orders[col] = orders[col].astype('category')

for col in cat_cols_products:
    products[col] = products[col].astype('category')

for col in cat_cols_returns:
    returns[col] = returns[col].astype('category')

print("✅ Категориальные типы назначены!")


In [ ]:
# ================================================================
# Шаг 3: Создание новых признаков из дат
# Это поможет в анализе сезонности, динамики и поведения клиентов
# ================================================================

# --- В таблице orders ---
orders['order_year']    = orders['order_date'].dt.year
orders['order_month']   = orders['order_date'].dt.month
orders['order_month_name'] = orders['order_date'].dt.strftime('%b')  # Jan, Feb...
orders['order_weekday'] = orders['order_date'].dt.day_name()
orders['order_quarter'] = orders['order_date'].dt.quarter
orders['order_ym']      = orders['order_date'].dt.to_period('M')  # для группировки по месяцам

# Сезон (по месяцам — условно для индийского рынка)
def get_season(month):
    if month in [12, 1, 2]:
        return 'Зима'
    elif month in [3, 4, 5]:
        return 'Весна'
    elif month in [6, 7, 8]:
        return 'Лето'
    else:
        return 'Осень'

orders['season'] = orders['order_month'].apply(get_season)

# --- В таблице customers ---
customers['signup_year']  = customers['signup_date'].dt.year
customers['signup_month'] = customers['signup_date'].dt.month

print("✅ Новые признаки созданы!")
print("   Новые столбцы в orders:", [c for c in orders.columns if c not in ['order_id','customer_id','order_date','order_status','payment_method','sales_channel','gross_amount','discount_amount','shipping_fee','final_amount','delivered_date']])


In [ ]:
# ================================================================
# Шаг 4: Добавляем маржинальность к продуктам
# margin = mrp - cost_price (грубая маржа)
# margin_pct = маржа / mrp * 100
# ================================================================

products['margin']     = products['mrp'] - products['cost_price']
products['margin_pct'] = (products['margin'] / products['mrp'] * 100).round(1)

print("✅ Маржинальность рассчитана!")
print(products[['product_name', 'mrp', 'cost_price', 'margin', 'margin_pct']].head(5).to_string(index=False))


In [ ]:
# ================================================================
# Шаг 5: Создаём главную аналитическую таблицу
# Объединяем все таблицы в одну для удобства анализа
#
# Логика JOIN-ов:
# order_items — центральная таблица (каждая строка = один товар в заказе)
# + orders    — информация о заказе (дата, статус, сумма)
# + customers — информация о покупателе
# + products  — информация о товаре
# ================================================================

main_df = (
    order_items
    .merge(orders,    on='order_id',    how='left')  # добавляем инфо о заказе
    .merge(customers, on='customer_id', how='left')  # добавляем инфо о клиенте
    .merge(products,  on='product_id',  how='left')  # добавляем инфо о товаре
)

print(f"✅ Главная таблица создана: {main_df.shape[0]} строк × {main_df.shape[1]} столбцов")
print("\nПервые 2 строки (выборочные столбцы):")
display(main_df[['order_item_id','order_date','customer_name','gender','product_name','category','quantity','item_total']].head(2))


---
<a id='eda2'></a>
## 6. 📊 Исследовательский анализ данных (EDA)

Теперь самое интересное — смотрим на данные глазами бизнеса.  
Каждый график — это ответ на конкретный вопрос.


In [ ]:
# ================================================================
# График 1: Динамика выручки по месяцам
# Вопрос: Как менялась выручка со временем? Есть ли рост или падение?
# ================================================================

monthly_revenue = (
    orders[orders['order_status'] != 'Cancelled']  # убираем отменённые заказы
    .groupby('order_ym')['final_amount']
    .sum()
    .reset_index()
)
monthly_revenue['order_ym_str'] = monthly_revenue['order_ym'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(
    monthly_revenue['order_ym_str'],
    monthly_revenue['final_amount'],
    marker='o', color='#2E86AB', linewidth=2.5, markersize=6
)

# Добавляем скользящее среднее (окно 3 месяца) — сглаживает шум
rolling = monthly_revenue['final_amount'].rolling(3, center=True).mean()
ax.plot(
    monthly_revenue['order_ym_str'],
    rolling,
    linestyle='--', color='#E84855', alpha=0.8, linewidth=2, label='Скол. среднее (3 мес.)'
)

ax.set_title('💰 Динамика выручки по месяцам', fontsize=14, pad=15)
ax.set_xlabel('Месяц')
ax.set_ylabel('Выручка (₹)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
ax.tick_params(axis='x', rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig('charts/01_monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n📌 Вывод:")
print(f"   Общая выручка (без отмен): ₹{monthly_revenue['final_amount'].sum():,.0f}")
print(f"   Средняя выручка в месяц:  ₹{monthly_revenue['final_amount'].mean():,.0f}")
print(f"   Пиковый месяц: {monthly_revenue.loc[monthly_revenue['final_amount'].idxmax(), 'order_ym_str']}")
print(f"   Мин. выручка:  {monthly_revenue.loc[monthly_revenue['final_amount'].idxmin(), 'order_ym_str']}")


In [ ]:
# ================================================================
# График 2: Топ-5 категорий товаров по выручке
# Вопрос: Что продаётся лучше всего?
# ================================================================

category_revenue = (
    main_df[main_df['order_status'] != 'Cancelled']
    .groupby('category')['item_total']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Левый: барплот выручки
colors = sns.color_palette('Set2', len(category_revenue))
bars = axes[0].barh(
    category_revenue['category'][::-1],
    category_revenue['item_total'][::-1],
    color=colors[::-1]
)
axes[0].set_title('Выручка по категориям товаров', fontsize=13)
axes[0].set_xlabel('Сумма (₹)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
for bar, val in zip(bars, category_revenue['item_total'][::-1]):
    axes[0].text(val + 500, bar.get_y() + bar.get_height()/2,
                 f'₹{val/1000:.1f}K', va='center', fontsize=10)

# Правый: доля каждой категории (пирог)
axes[1].pie(
    category_revenue['item_total'],
    labels=category_revenue['category'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    pctdistance=0.75
)
axes[1].set_title('Доля категорий в выручке', fontsize=13)

plt.suptitle('📦 Анализ категорий товаров', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('charts/02_category_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 Вывод:")
for i, row in category_revenue.iterrows():
    share = row['item_total'] / category_revenue['item_total'].sum() * 100
    print(f"   {row['category']:<20} ₹{row['item_total']:>8,.0f}  ({share:.1f}%)")


In [ ]:
# ================================================================
# График 3: Каналы продаж и способы оплаты
# Вопрос: Где и как покупают наши клиентки?
# ================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Канал продаж
channel_data = (
    orders[orders['order_status'] != 'Cancelled']
    .groupby('sales_channel')
    .agg(orders_count=('order_id', 'count'), revenue=('final_amount', 'sum'))
    .reset_index()
    .sort_values('revenue', ascending=False)
)

colors_ch = sns.color_palette('Set2', len(channel_data))
axes[0].bar(channel_data['sales_channel'], channel_data['revenue'], color=colors_ch)
axes[0].set_title('Выручка по каналам продаж', fontsize=13)
axes[0].set_ylabel('Выручка (₹)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
for i, (_, row) in enumerate(channel_data.iterrows()):
    axes[0].text(i, row['revenue'] + 500, f"{row['orders_count']} зак.", ha='center', fontsize=10)

# Способ оплаты
pay_data = (
    orders[orders['order_status'] != 'Cancelled']
    .groupby('payment_method')['final_amount']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

axes[1].barh(pay_data['payment_method'][::-1], pay_data['final_amount'][::-1],
             color=sns.color_palette('Set2', len(pay_data)))
axes[1].set_title('Выручка по способам оплаты', fontsize=13)
axes[1].set_xlabel('Выручка (₹)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))

plt.suptitle('🛒 Каналы и способы оплаты', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('charts/03_channels_payment.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ================================================================
# График 4: Клиентки по полу и возрасту
# Вопрос: Кто наша аудитория?
# ================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Гендерное распределение
gender_data = customers['gender'].value_counts()
colors_g = ['#FF9BB5', '#85C1E9']
axes[0].pie(
    gender_data.values,
    labels=gender_data.index,
    autopct='%1.1f%%',
    colors=colors_g,
    startangle=90
)
axes[0].set_title('Распределение по полу', fontsize=13)

# Возрастные группы
age_data = (
    customers.groupby('age_group').size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
# Упорядочиваем по логике возраста
age_order = ['18-24', '25-34', '35-44', '45-54', '55+']
age_data['age_group'] = pd.Categorical(age_data['age_group'], categories=age_order, ordered=True)
age_data = age_data.sort_values('age_group')

bars = axes[1].bar(
    age_data['age_group'].astype(str),
    age_data['count'],
    color=sns.color_palette('Set2', len(age_data))
)
axes[1].set_title('Клиентки по возрастным группам', fontsize=13)
axes[1].set_ylabel('Количество клиентов')
for bar in bars:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(int(bar.get_height())), ha='center', fontsize=11)

plt.suptitle('👩 Портрет нашей аудитории', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('charts/04_audience.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ================================================================
# График 5: Топ-5 штатов по выручке (географический анализ)
# Вопрос: Откуда приходят наши лучшие клиентки?
# ================================================================

geo_data = (
    main_df[main_df['order_status'] != 'Cancelled']
    .groupby('state')['item_total']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(geo_data['state'][::-1], geo_data['item_total'][::-1],
               color=sns.color_palette('Set2', len(geo_data)))
ax.set_title('🗺️ Топ-10 штатов по выручке', fontsize=14)
ax.set_xlabel('Выручка (₹)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K'))
for bar, val in zip(bars, geo_data['item_total'][::-1]):
    ax.text(val + 200, bar.get_y() + bar.get_height()/2,
            f'₹{val/1000:.1f}K', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('charts/05_geo_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 Вывод: Концентрация клиентов")
print(geo_data.to_string(index=False))


In [ ]:
# ================================================================
# График 6: Средний чек по каналу привлечения и полу
# Вопрос: Клиенты из какого канала тратят больше?
# ================================================================

acq_data = (
    main_df[main_df['order_status'] != 'Cancelled']
    .groupby('acquisition_channel')
    .agg(
        total_revenue=('item_total', 'sum'),
        orders=('order_id', 'nunique'),
        customers=('customer_id', 'nunique')
    )
    .reset_index()
)
acq_data['avg_order_value'] = (acq_data['total_revenue'] / acq_data['orders']).round(0)
acq_data = acq_data.sort_values('avg_order_value', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Средний чек по каналу
bars = axes[0].bar(
    acq_data['acquisition_channel'],
    acq_data['avg_order_value'],
    color=sns.color_palette('Set2', len(acq_data))
)
axes[0].set_title('Средний чек по каналу привлечения', fontsize=13)
axes[0].set_ylabel('Средний чек (₹)')
axes[0].tick_params(axis='x', rotation=30)
for bar, val in zip(bars, acq_data['avg_order_value']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'₹{val:.0f}', ha='center', fontsize=10)

# Число клиентов по каналу
bars2 = axes[1].bar(
    acq_data['acquisition_channel'],
    acq_data['customers'],
    color=sns.color_palette('Set2', len(acq_data))
)
axes[1].set_title('Число клиентов по каналу привлечения', fontsize=13)
axes[1].set_ylabel('Клиентов')
axes[1].tick_params(axis='x', rotation=30)
for bar, val in zip(bars2, acq_data['customers']):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(int(val)), ha='center', fontsize=10)

plt.suptitle('📣 Эффективность каналов привлечения', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('charts/06_acquisition_channels.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ================================================================
# График 7: Корреляционный анализ числовых переменных
# Вопрос: Есть ли связи между числовыми метриками?
# ================================================================

# Берём числовые столбцы из главной таблицы
num_cols = ['quantity', 'unit_price', 'discount_pct', 'item_total',
            'gross_amount', 'discount_amount', 'shipping_fee', 'final_amount']

corr_matrix = main_df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # показываем только нижний треугольник
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    ax=ax,
    linewidths=0.5,
    vmin=-1, vmax=1
)
ax.set_title('🔗 Корреляционная матрица числовых переменных', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('charts/07_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 Вывод: Ключевые корреляции")
print("   Сильная положительная: unit_price ↔ item_total (чем дороже товар — тем больше сумма позиции)")
print("   Сильная положительная: gross_amount ↔ final_amount (логично — это близкие метрики)")
print("   Обратите внимание: discount_pct слабо связан с item_total — скидки не сильно влияют на итоговую сумму")


---
<a id='advanced'></a>
## 7. 🔬 Углублённая аналитика

В этом разделе я выбрала три направления:
1. **RFM-анализ** — сегментация клиентов по ценности
2. **Анализ возвратов** — проблемные товары и причины
3. **Анализ отзывов** — связь рейтинга с поведением клиентов


### 7.1 🏷️ RFM-анализ: Кто наши лучшие клиентки?

RFM — это классический метод сегментации клиентов по трём параметрам:
- **R (Recency)** — когда последний раз покупала? (меньше дней = лучше)
- **F (Frequency)** — как часто покупает? (больше = лучше)  
- **M (Monetary)** — сколько потратила? (больше = лучше)


In [ ]:
# ================================================================
# RFM-анализ
# Берём только доставленные заказы (статус Delivered)
# Дата среза — максимальная дата в данных + 1 день
# ================================================================

delivered_orders = orders[orders['order_status'] == 'Delivered'].copy()
snapshot_date = delivered_orders['order_date'].max() + pd.Timedelta(days=1)

print(f"Дата среза для расчёта Recency: {snapshot_date.date()}")
print(f"Заказов для анализа: {len(delivered_orders)}")

rfm = (
    delivered_orders
    .groupby('customer_id')
    .agg(
        last_order_date=('order_date', 'max'),
        frequency=('order_id', 'count'),
        monetary=('final_amount', 'sum')
    )
    .reset_index()
)

rfm['recency'] = (snapshot_date - rfm['last_order_date']).dt.days
rfm = rfm.drop('last_order_date', axis=1)

print(f"\nRFM-таблица создана: {rfm.shape[0]} клиентов")
display(rfm.describe().round(1))


In [ ]:
# ================================================================
# Присваиваем RFM-баллы (1-5, где 5 — лучший)
# Для Recency: меньше дней = выше балл (поэтому ranking обратный)
# ================================================================

rfm['R_score'] = pd.qcut(rfm['recency'],   q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_score'] = pd.qcut(rfm['monetary'],  q=5, labels=[1,2,3,4,5]).astype(int)

rfm['RFM_score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)
rfm['RFM_total'] = rfm['R_score'] + rfm['F_score'] + rfm['M_score']

# Сегменты клиентов на основе баллов
def rfm_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4:
        return '⭐ Чемпионы'
    elif r >= 3 and f >= 3:
        return '💛 Лояльные'
    elif r >= 4 and f <= 2:
        return '🆕 Новые клиентки'
    elif r <= 2 and f >= 3 and m >= 3:
        return '😴 Засыпающие'
    elif r <= 2 and f <= 2:
        return '💤 Потерянные'
    elif m >= 4:
        return '💎 Крупные покупки'
    else:
        return '🟡 Обычные'

rfm['segment'] = rfm.apply(rfm_segment, axis=1)

segment_stats = rfm.groupby('segment').agg(
    count=('customer_id', 'count'),
    avg_recency=('recency', 'mean'),
    avg_frequency=('frequency', 'mean'),
    avg_monetary=('monetary', 'mean')
).round(1).sort_values('count', ascending=False)

print("📊 RFM-сегменты:")
display(segment_stats)


In [ ]:
# ================================================================
# Визуализация RFM-сегментов
# ================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Размер сегментов
seg_counts = rfm['segment'].value_counts()
colors_rfm = sns.color_palette('Set2', len(seg_counts))

axes[0].barh(seg_counts.index, seg_counts.values, color=colors_rfm)
axes[0].set_title('Число клиентов по сегментам', fontsize=13)
axes[0].set_xlabel('Количество клиентов')
for i, (seg, cnt) in enumerate(seg_counts.items()):
    axes[0].text(cnt + 0.5, i, str(cnt), va='center', fontsize=11)

# Средняя выручка по сегменту
seg_revenue = rfm.groupby('segment')['monetary'].mean().sort_values(ascending=True)
axes[1].barh(seg_revenue.index, seg_revenue.values,
             color=[colors_rfm[list(seg_counts.index).index(s)] for s in seg_revenue.index])
axes[1].set_title('Средняя выручка на клиента по сегменту', fontsize=13)
axes[1].set_xlabel('Средняя выручка (₹)')
for i, (seg, val) in enumerate(seg_revenue.items()):
    axes[1].text(val + 10, i, f'₹{val:.0f}', va='center', fontsize=10)

plt.suptitle('🏷️ RFM-сегментация клиентов', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('charts/08_rfm_segments.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 Вывод по RFM:")
champions = rfm[rfm['segment'] == '⭐ Чемпионы']
print(f"   Чемпионов: {len(champions)} клиентов ({len(champions)/len(rfm)*100:.1f}%)")
print(f"   Их средняя выручка: ₹{champions['monetary'].mean():.0f} vs общая средняя ₹{rfm['monetary'].mean():.0f}")


### 7.2 📦 Анализ возвратов

Это важный раздел — возвраты напрямую влияют на прибыльность и репутацию бренда.  
Давайте разберёмся, что возвращают и почему.


In [ ]:
# ================================================================
# Анализ возвратов
# Объединяем returns + products + orders для полной картины
# ================================================================

returns_full = (
    returns
    .merge(products[['product_id', 'product_name', 'category', 'mrp']], on='product_id', how='left')
    .merge(orders[['order_id', 'customer_id', 'final_amount']], on='order_id', how='left')
)

print(f"Всего возвратов: {len(returns_full)}")
print(f"Уникальных заказов с возвратами: {returns_full['order_id'].nunique()}")

# Доля заказов с возвратами
total_orders = orders[orders['order_status'] == 'Delivered']['order_id'].nunique()
return_rate = returns_full['order_id'].nunique() / total_orders * 100
print(f"Общий показатель возвратов: {return_rate:.1f}%")

# Причины возвратов
return_reasons = returns_full['return_reason'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Причины
axes[0].barh(return_reasons.index[::-1], return_reasons.values[::-1],
             color=sns.color_palette('Set2', len(return_reasons)))
axes[0].set_title('Причины возвратов', fontsize=13)
axes[0].set_xlabel('Количество возвратов')
for i, (reason, cnt) in enumerate(zip(return_reasons.index[::-1], return_reasons.values[::-1])):
    axes[0].text(cnt + 0.2, i, str(cnt), va='center', fontsize=11)

# Категории товаров с возвратами
cat_returns = (
    returns_full.groupby('category').size()
    .sort_values(ascending=False)
    .reset_index(name='return_count')
)
# Считаем общее кол-во продаж по категории для % возвратов
cat_sales = (
    main_df[main_df['order_status'] == 'Delivered']
    .groupby('category')['order_id']
    .nunique()
    .reset_index(name='total_orders')
)
cat_returns = cat_returns.merge(cat_sales, on='category', how='left')
cat_returns['return_rate'] = (cat_returns['return_count'] / cat_returns['total_orders'] * 100).round(1)

colors_cr = sns.color_palette('Set2', len(cat_returns))
axes[1].bar(cat_returns['category'], cat_returns['return_rate'], color=colors_cr)
axes[1].set_title('% возвратов по категориям', fontsize=13)
axes[1].set_ylabel('Доля возвратов (%)')
axes[1].tick_params(axis='x', rotation=30)
for i, row in cat_returns.iterrows():
    axes[1].text(i, row['return_rate'] + 0.1, f"{row['return_rate']}%", ha='center', fontsize=10)

plt.suptitle('📦 Анализ возвратов', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('charts/09_returns_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 Главные причины возвратов:")
for reason, cnt in return_reasons.items():
    print(f"   {reason}: {cnt} случаев ({cnt/len(returns_full)*100:.1f}%)")


### 7.3 ⭐ Анализ отзывов и рейтингов

Мне было интересно проверить: влияет ли рейтинг товара на вероятность его возврата?  
Интуитивно — если товар плохой, его и оценивают ниже, и возвращают чаще.


In [ ]:
# ================================================================
# Анализ отзывов
# ================================================================

reviews_full = (
    reviews
    .merge(products[['product_id', 'product_name', 'category']], on='product_id', how='left')
    .merge(orders[['order_id', 'order_status']], on='order_id', how='left')
)

print(f"Всего отзывов: {len(reviews_full)}")
print(f"Средний рейтинг по всем продуктам: {reviews_full['rating'].mean():.2f} ⭐")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Распределение рейтингов
rating_dist = reviews_full['rating'].value_counts().sort_index()
colors_r = ['#E74C3C', '#E67E22', '#F1C40F', '#2ECC71', '#1ABC9C']
axes[0].bar(rating_dist.index, rating_dist.values, color=colors_r)
axes[0].set_title('Распределение оценок', fontsize=13)
axes[0].set_xlabel('Рейтинг (звёзды)')
axes[0].set_ylabel('Количество отзывов')
for i, (rating, cnt) in enumerate(rating_dist.items()):
    axes[0].text(rating, cnt + 1, str(cnt), ha='center', fontsize=11)

# Средний рейтинг по категории
cat_rating = (
    reviews_full.groupby('category')['rating']
    .mean()
    .sort_values(ascending=True)
    .reset_index()
)
axes[1].barh(cat_rating['category'], cat_rating['rating'],
             color=sns.color_palette('Set2', len(cat_rating)))
axes[1].set_title('Средний рейтинг по категориям', fontsize=13)
axes[1].set_xlabel('Средний рейтинг')
axes[1].set_xlim(0, 5)
for i, row in cat_rating.iterrows():
    axes[1].text(row['rating'] + 0.05, i, f'{row["rating"]:.2f}', va='center', fontsize=11)

# Связь рейтинга с возвратами
returned_orders = set(returns['order_id'])
reviews_full['was_returned'] = reviews_full['order_id'].isin(returned_orders)

rating_return = reviews_full.groupby('rating')['was_returned'].mean() * 100

axes[2].bar(rating_return.index, rating_return.values,
            color=['#E74C3C' if r < 3 else '#2ECC71' for r in rating_return.index])
axes[2].set_title('% возвратов по рейтингу отзыва', fontsize=13)
axes[2].set_xlabel('Рейтинг')
axes[2].set_ylabel('Доля заказов с возвратом (%)')
for rating, val in rating_return.items():
    axes[2].text(rating, val + 0.2, f'{val:.1f}%', ha='center', fontsize=11)

plt.suptitle('⭐ Анализ отзывов и рейтингов', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('charts/10_reviews_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n📌 Интересное наблюдение:")
print(f"   Среди заказов с оценкой 1-2: {rating_return[rating_return.index <= 2].mean():.1f}% было возвращено")
print(f"   Среди заказов с оценкой 4-5: {rating_return[rating_return.index >= 4].mean():.1f}% было возвращено")


---
<a id='insights'></a>
## 8. 💡 Ключевые инсайты

По результатам анализа я выделила несколько важных наблюдений.  
Пишу осторожно, с оговорками — данные синтетические, и выводы требуют проверки на реальных данных.

---

### Инсайт 1: Выручка неравномерна по месяцам
Мы заметили заметные колебания месячной выручки — возможно, это связано с сезонностью (уход за кожей летом и зимой может сильно различаться). Стоит дополнительно проверить гипотезу о том, что пиковые месяцы совпадают с праздниками или сменой сезона.

### Инсайт 2: Сыворотки — самая выручаемая категория
Категория Serum стабильно занимает первое место по выручке. Интересно, что клиентки в возрасте 25-34 года, судя по всему, активнее всего покупают именно уходовые сыворотки — это аудитория, которая уже разбирается в косметике и готова платить за эффективные ингредиенты.

### Инсайт 3: RFM показал, что большинство клиентов — «обычные»
Только небольшая доля клиентов попала в сегмент «Чемпионы» с высокими баллами по всем трём метрикам. Это нормально для большинства брендов, но говорит о потенциале для удержания: «Лояльных» клиентов можно перевести в «Чемпионов» через персонализацию.

### Инсайт 4: Основная причина возвратов — позднее получение товара
Это не проблема качества продукта, а проблема логистики. Возможно, стоит пересмотреть партнёра по доставке или улучшить управление ожиданиями клиентов.

### Инсайт 5: Низкий рейтинг коррелирует с возвратами
Заказы, которым поставили 1-2 звезды, возвращают чаще, чем те, которым поставили 4-5 звезд. Это логично, но важно как подтверждение: рейтинговая система работает как сигнал.

### Инсайт 6: Instagram и YouTube дают клиентов со схожим средним чеком
Мы заметили, что средний чек по разным каналам привлечения отличается незначительно. Возможно, это специфика синтетических данных — на реальных данных разница была бы больше.

### Инсайт 7: Мобильное приложение — растущий канал продаж
Mobile App показывает хорошие результаты и, возможно, более удобен для повторных покупок — это стоит проверить, посмотрев на Frequency у клиентов из этого канала.

---
> ⚠️ *Напоминаю: все цифры из синтетического датасета. На реальных данных картина может быть принципиально иной.*


---
<a id='recommendations'></a>
## 9. 📋 Бизнес-рекомендации

На основе анализа я сформулировала конкретные рекомендации.  
Каждая привязана к данным и содержит предполагаемый эффект.

---

### Рекомендация 1: Реактивировать «Засыпающих» клиентов
**Что сделать:** Запустить email/push-кампанию для сегмента «Засыпающие» (R_score ≤ 2, F ≥ 3) с персональной скидкой 10-15%.  
**Почему:** Эти клиентки раньше покупали регулярно, значит, они знают бренд и доверяют ему — барьер для возврата ниже, чем у новых.  
**Потенциальный эффект:** Даже возврат 20% «Засыпающих» может дать заметный прирост к месячной выручке.

### Рекомендация 2: Сфокусироваться на категории Serum
**Что сделать:** Расширить ассортимент сывороток (новые ингредиенты, форматы), выделить отдельный блок на сайте и в приложении.  
**Почему:** Serum — ведущая категория по выручке с высокой маржой.  
**Потенциальный эффект:** Кросс-продажи (сыворотка + тонер + крем) могут увеличить средний чек.

### Рекомендация 3: Решить проблему с доставкой
**Что сделать:** Провести аудит текущих сроков доставки, рассмотреть альтернативных логистических партнёров, добавить трекинг заказа в приложение.  
**Почему:** «Позднее получение» — главная причина возвратов. Это потеря денег и доверия.  
**Потенциальный эффект:** Снижение возвратов хотя бы на 30% позволит сохранить выручку.

### Рекомендация 4: Вознаграждать «Чемпионов»
**Что сделать:** Создать закрытую программу лояльности (ранний доступ к новинкам, бесплатная доставка, персональный консультант).  
**Почему:** Это самые ценные клиентки — они тратят значительно больше среднего и возвращаются снова.  
**Потенциальный эффект:** Удержание «Чемпионов» значительно дешевле привлечения новых клиентов.

### Рекомендация 5: Мониторить товары с низким рейтингом
**Что сделать:** Настроить автоматический алерт — если средний рейтинг продукта падает ниже 3.0 за последние 30 дней, это сигнал для команды качества.  
**Почему:** Низкий рейтинг → выше возвраты → выше расходы на логистику и хуже репутация.  
**Потенциальный эффект:** Раннее обнаружение проблем снижает риски репутационного ущерба.


---
<a id='conclusion'></a>
## 10. 🏁 Заключение и следующие шаги

### Что я сделала в этом проекте:
- ✅ Загрузила и очистила 6 таблиц данных D2C-бренда косметики
- ✅ Связала таблицы через JOIN и создала единую аналитическую базу
- ✅ Провела разведочный анализ с 10+ визуализациями
- ✅ Сегментировала клиентов с помощью RFM-анализа
- ✅ Проанализировала возвраты и отзывы
- ✅ Сформулировала конкретные бизнес-рекомендации

### Ограничения:
- Данные синтетические и не отражают реальную бизнес-ситуацию
- Отсутствует информация о маркетинговых расходах (нельзя посчитать CAC и ROI по каналам)
- Нет данных о сессиях на сайте (нельзя посчитать конверсию воронки)

### Следующие шаги (если бы это был реальный проект):
1. **Когортный анализ** — посмотреть Retention по когортам (месяц первой покупки)
2. **Анализ корзины (Market Basket Analysis)** — какие товары часто покупают вместе?
3. **Прогнозирование оттока** — попробовать логистическую регрессию или дерево решений
4. **A/B тесты** — проверить гипотезы из раздела рекомендаций на реальных данных
5. **Подключение Power BI** — сделать интерактивный дашборд для бизнес-команды

---
> *Это мой первый большой аналитический проект. Буду рада любой обратной связи!*  
> *GitHub: [ваша ссылка] | LinkedIn: [ваша ссылка]*
